# LTX-2 C2V — Latent Trajectory Analysis
### exp_021 · run_0004 · Stage-1 denoising trajectory

This notebook inspects the **full Stage-1 denoising trajectory** logged by `exp_021` at three distinct internal representations of LTX-2:

| Level | Tensor | Shape | What it encodes |
|-------|--------|-------|-----------------|
| **1 — VAE Latent** | `z_t` | `[S=40, C=128, F'=16, H'=16, W'=24]` | Noisy video in VAE latent space — how the video crystallises over 40 steps |
| **2 — Velocity Field** | `v_pred` | `[S=40, C=128, F'=16, H'=16, W'=24]` | Flow-matching velocity — the force pushing each frame toward the target |
| **3 — Transformer Activations** | `hidden_states` | `{step: {block: [2, 6144, 4096]}}` | Internal representations inside LTX-2's DiT transformer blocks |

---

### Terminology used throughout this notebook

| Term | Symbol | Definition |
|------|--------|-----------|
| **Latent frame** | `p ∈ [0..15]` | Temporal unit in VAE latent space. 16 latent frames compress 121 pixel-frames (8× temporal compression). Each latent frame ≈ 8 pixel-frames ≈ 0.33 s at 24 fps. |
| **Denoising step** | `τ ∈ [0..39]` | Stage in the 40-step iterative denoising. τ=0: pure Gaussian noise. τ=39: fully denoised clean latent `z₀`. |
| **VAE channel** | `c ∈ [0..127]` | One of 128 channels in the VAE latent. Always aggregated via L2 norm — never shown individually. |
| **Transformer block** | `L ∈ {12,24,35,47}` | One of LTX-2's 48 transformer blocks. We probe 4 depths: L12 (25%), L24 (50%), L35 (75%), L47 (98%). This is what "layer" means in Level 3. |
| **Token** | `(p, h, w)` | Atomic unit of the transformer. LTX-2 packs all spatiotemporal positions into one flat sequence: 16×16×24 = **6,144 tokens** total. |
| **Token embedding** | `h^L(p,h,w)` | The 4096-dim vector that transformer block `L` outputs for token at latent-frame `p`, height `h`, width `w`. |
| **Frame mean-embedding** | `h̄^L(p)` | Average of the 16×24=384 token embeddings within latent frame `p` at block `L` → one 4096-dim summary vector per frame. |
| **Dissolve frame** | `p*` | Predicted latent frame where the content transition occurs (estimated from curvature peak of `z₀`). |

---

### Conditioning geometry (this run)

```
Latent frames:  p=0  p=1  p=2  p=3  |  p=4  p=5 ... p=11  |  p=12  p=13  p=14  p=15
                ←  start-clip (cyan) →  ← free middle (white) →  ← end-clip (magenta) →
                  K_LAT=4 frames          8 model-generated frames    K_LAT=4 frames
```

- **Cyan** (`p=0..3`): fixed to the last 4 latent frames of the start clip — the model must reconstruct these exactly.
- **Free middle** (`p=4..11`): freely generated — this is where the transition must occur.
- **Magenta** (`p=12..15`): fixed to the first 4 latent frames of the end clip.

**Primary hypothesis:** When start/end clips are semantically different (classes 6, 8), LTX-2 places a hard **dissolve cut** at some frame `p*` in the free-middle region, visible as a curvature spike in `z₀` (Level 1), a persistent velocity stripe at `p*` (Level 2), and a block-diagonal similarity matrix in deep transformer blocks (Level 3).

In [ ]:
import sys, pathlib, base64, warnings, re
import numpy as np
import torch
import yaml
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
from IPython.display import HTML, display

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({"figure.dpi": 110, "font.size": 9})

# ── CONFIG ────────────────────────────────────────────────────────────────────
TRAJ_DIR = pathlib.Path(
    "/workspace/diffusion-research/outputs/videos/exp_021_trajectory_logging/run_0004"
)

# LTX-2 constants (from exp_021 config)
LTX_TEMPORAL_SCALE = 8
VIDEO_FPS          = 24.0
C_LATENT           = 128      # VAE channel count
F_PRIME            = 16       # latent temporal frames  (121 pixel frames)
H_PRIME            = 16       # latent height           (512 px / 32)
W_PRIME            = 24       # latent width            (768 px / 32)
S_STEPS            = 40       # Stage-1 denoising steps
HIDDEN_DIM         = 4096     # transformer hidden size

# Conditioning geometry  (num_clip_frames=25, num_frames=121)
K_LAT   = (25  - 1) // LTX_TEMPORAL_SCALE + 1                          # 4
END_IDX = (F_PRIME * LTX_TEMPORAL_SCALE - 7) // LTX_TEMPORAL_SCALE - K_LAT + K_LAT  # computed below
END_IDX = F_PRIME - K_LAT                                               # 12

# Semantic class colour palette (from exp_022)
CLASS_COLORS = {"1": "#2196F3", "2": "#4CAF50", "5": "#FF9800", "6": "#F44336", "8": "#9C27B0"}
CLASS_LABELS = {
    "1": "similar ctx / cat / motion",
    "2": "similar ctx / cat / diff motion",
    "5": "diff ctx / similar cat / similar motion",
    "6": "diff ctx / similar cat / diff motion",
    "8": "diff ctx / cat / motion",
}

print(f"TRAJ_DIR      : {TRAJ_DIR}")
print(f"Latent shape  : C={C_LATENT}, F'={F_PRIME}, H'={H_PRIME}, W'={W_PRIME}")
print(f"Conditioning  : start p=[0..{K_LAT-1}]  |  free p=[{K_LAT}..{END_IDX-1}]  |  end p=[{END_IDX}..{END_IDX+K_LAT-1}]")
print(f"Hidden shape  : [2, {F_PRIME*H_PRIME*W_PRIME}, {HIDDEN_DIM}]  (batch=2 for CFG)")

In [ ]:
# ── Feature Computation Utilities ────────────────────────────────────────────
# All functions operate on tensors in their already-unpacked form:
#   z_t   : [S, C, F', H', W']
#   z_final: [C, F', H', W']

def frame_norm(t: torch.Tensor) -> torch.Tensor:
    """Per-frame L2 norm: [S, C, F', H', W'] → [S, F'] (noise-level proxy)."""
    return t.pow(2).sum(dim=(1, 3, 4)).sqrt()

def frame_norm_1d(t: torch.Tensor) -> torch.Tensor:
    """Per-frame L2 norm for final latent: [C, F', H', W'] → [F']."""
    return t.pow(2).sum(dim=(0, 2, 3)).sqrt()

def spatial_diff(t: torch.Tensor):
    """Temporal differences along the F' axis.

    Args:
        t: [S, C, F', H', W'] or [C, F', H', W']
    Returns:
        speed     [S, F'-1] or [F'-1]   ‖Δ_p z‖
        curvature [S, F'-2] or [F'-2]   ‖Δ²_p z‖      ← dissolve bend signal
        angular   [S, F'-2] or [F'-2]   cos(Δ_p, Δ_{p+1})  ← direction flip
    """
    is_5d = (t.ndim == 5)
    if not is_5d:
        t = t.unsqueeze(0)

    dz    = t[:, :, 1:, :, :] - t[:, :, :-1, :, :]          # [S, C, F'-1, H', W']
    speed = dz.pow(2).sum(dim=(1, 3, 4)).sqrt()              # [S, F'-1]

    d2z  = dz[:, :, 1:, :, :] - dz[:, :, :-1, :, :]         # [S, C, F'-2, H', W']
    curv = d2z.pow(2).sum(dim=(1, 3, 4)).sqrt()              # [S, F'-2]

    dz_flat = dz.permute(0, 2, 1, 3, 4).flatten(2)           # [S, F'-1, C*H'*W']
    dz_unit = dz_flat / dz_flat.norm(dim=2, keepdim=True).clamp(min=1e-8)
    ang = (dz_unit[:, :-1, :] * dz_unit[:, 1:, :]).sum(dim=2)  # [S, F'-2]

    if not is_5d:
        return speed.squeeze(0), curv.squeeze(0), ang.squeeze(0)
    return speed, curv, ang


def compute_features(data: dict) -> dict:
    """Extract all geometric features from one trajectory .pt dict."""
    z  = data["z_t"].float()      # [S, C, F', H', W']
    v  = data["v_pred"].float()   # [S, C, F', H', W']
    z0 = data["z_final"].float()  # [C, F', H', W']
    S, _, F = z.shape[0], z.shape[1], z.shape[2]
    ts = np.array(data["timesteps"])   # [S] — descending from ~1000 to 0

    # Level 1: VAE latent features
    norm_z                     = frame_norm(z)           # [S, F']
    speed_z, curv_z, ang_z     = spatial_diff(z)        # [S,F'-1], [S,F'-2], [S,F'-2]

    # Level 2: Velocity field features
    pred_mag                   = frame_norm(v)           # [S, F']
    _, pred_curv, _            = spatial_diff(v)         # [S, F'-2]

    # Per-frame denoising step-size (how much each frame changes each step)
    z_next      = torch.cat([z[1:], z0.unsqueeze(0)], dim=0)  # [S, C, F', H', W']
    step_size_z = frame_norm(z_next - z)                        # [S, F']

    # Final clean-latent z_0 signals (most diagnostic for dissolve location)
    norm_z0                       = frame_norm_1d(z0)        # [F']
    speed_z0, curv_z0, ang_z0     = spatial_diff(z0)         # [F'-1], [F'-2], [F'-2]
    pred_mag0                     = pred_mag[-1]              # [F'] — last step velocity

    # Dissolve scalars (adapted from exp_022)
    c0 = curv_z0.numpy()
    dissolve_frame    = int(np.argmax(c0)) + 1            # +1: curvature is centred
    dissolve_strength = float(c0.max() / (c0.mean() + 1e-8))
    angular_min       = float(ang_z0.numpy().min())

    # Free-middle–restricted dissolve frame (p = K_LAT .. END_IDX-1)
    fi_s = K_LAT - 1
    fi_e = END_IDX - 1
    free_c = c0[fi_s:fi_e]
    dissolve_frame_free = int(np.argmax(free_c)) + fi_s + 1 if len(free_c) > 0 else dissolve_frame

    # Dissolve-frame trajectory: argmax(curvature) per denoising step → shows commitment
    dissolve_by_step = np.argmax(curv_z.numpy(), axis=1) + 1  # [S]

    return {
        # 2-D trajectory matrices [S, *]
        "norm_z":       norm_z.numpy(),
        "speed_z":      speed_z.numpy(),
        "curvature_z":  curv_z.numpy(),
        "angular_z":    ang_z.numpy(),
        "pred_mag":     pred_mag.numpy(),
        "pred_curv":    pred_curv.numpy(),
        "step_size_z":  step_size_z.numpy(),
        # 1-D final-latent vectors [*]
        "norm_z0":      norm_z0.numpy(),
        "speed_z0":     speed_z0.numpy(),
        "curvature_z0": curv_z0.numpy(),
        "angular_z0":   ang_z0.numpy(),
        "pred_mag0":    pred_mag0.numpy(),
        # Dissolve scalars
        "dissolve_frame":      dissolve_frame,
        "dissolve_frame_free": dissolve_frame_free,
        "dissolve_strength":   dissolve_strength,
        "angular_min":         angular_min,
        "dissolve_by_step":    dissolve_by_step,
        "timesteps":           ts,
        "S": S, "F": F,
    }

print("Feature utilities ready.")

In [ ]:

def spatial_diff(t: torch.Tensor):
    """Temporal differences along the F' axis.

    Args:
        t: [S, C, F', H', W'] or [C, F', H', W']
    Returns:
        speed     [S, F'-1] or [F'-1]   ‖Δ_p z‖
        curvature [S, F'-2] or [F'-2]   ‖Δ²_p z‖      ← dissolve bend signal
        angular   [S, F'-2] or [F'-2]   cos(Δ_p, Δ_{p+1})  ← direction flip
    """
    is_5d = (t.ndim == 5)
    if not is_5d:
        t = t.unsqueeze(0)

    dz    = t[:, :, 1:, :, :] - t[:, :, :-1, :, :]          # [S, C, F'-1, H', W']
    speed = dz.pow(2).sum(dim=(1, 3, 4)).sqrt()              # [S, F'-1]

    d2z  = dz[:, :, 1:, :, :] - dz[:, :, :-1, :, :]         # [S, C, F'-2, H', W']
    curv = d2z.pow(2).sum(dim=(1, 3, 4)).sqrt()              # [S, F'-2]

    dz_flat = dz.permute(0, 2, 1, 3, 4).flatten(2)           # [S, F'-1, C*H'*W']
    dz_unit = dz_flat / dz_flat.norm(dim=2, keepdim=True).clamp(min=1e-8)
    ang = (dz_unit[:, :-1, :] * dz_unit[:, 1:, :]).sum(dim=2)  # [S, F'-2]

    if not is_5d:
        return speed.squeeze(0), curv.squeeze(0), ang.squeeze(0)
    return speed, curv, ang


def compute_features(data: dict) -> dict:
    """Extract all geometric features from one trajectory .pt dict."""
    z  = data["z_t"].float()      # [S, C, F', H', W']
    v  = data["v_pred"].float()   # [S, C, F', H', W']
    z0 = data["z_final"].float()  # [C, F', H', W']
    S, _, F = z.shape[0], z.shape[1], z.shape[2]
    ts = np.array(data["timesteps"])   # [S] — descending from ~1000 to 0

    # Level 1: VAE latent features
    norm_z                     = frame_norm(z)           # [S, F']
    speed_z, curv_z, ang_z     = spatial_diff(z)        # [S,F'-1], [S,F'-2], [S,F'-2]

    # Level 2: Velocity field features
    pred_mag                   = frame_norm(v)           # [S, F']
    _, pred_curv, _            = spatial_diff(v)         # [S, F'-2]

    # Per-frame denoising step-size (how much each frame changes each step)
    z_next      = torch.cat([z[1:], z0.unsqueeze(0)], dim=0)  # [S, C, F', H', W']
    step_size_z = frame_norm(z_next - z)                        # [S, F']

    # Final clean-latent z_0 signals (most diagnostic for dissolve location)
    norm_z0                       = frame_norm_1d(z0)        # [F']
    speed_z0, curv_z0, ang_z0     = spatial_diff(z0)         # [F'-1], [F'-2], [F'-2]
    pred_mag0                     = pred_mag[-1]              # [F'] — last step velocity

    # Dissolve scalars (adapted from exp_022)
    c0 = curv_z0.numpy()
    dissolve_frame    = int(np.argmax(c0)) + 1            # +1: curvature is centred
    dissolve_strength = float(c0.max() / (c0.mean() + 1e-8))
    angular_min       = float(ang_z0.numpy().min())

    # Free-middle–restricted dissolve frame (p = K_LAT .. END_IDX-1)
    fi_s = K_LAT - 1
    fi_e = END_IDX - 1
    free_c = c0[fi_s:fi_e]
    dissolve_frame_free = int(np.argmax(free_c)) + fi_s + 1 if len(free_c) > 0 else dissolve_frame

    # Dissolve-frame trajectory: argmax(curvature) per denoising step → shows commitment
    dissolve_by_step = np.argmax(curv_z.numpy(), axis=1) + 1  # [S]

    return {
        # 2-D trajectory matrices [S, *]
        "norm_z":       norm_z.numpy(),
        "speed_z":      speed_z.numpy(),
        "curvature_z":  curv_z.numpy(),
        "angular_z":    ang_z.numpy(),
        "pred_mag":     pred_mag.numpy(),
        "pred_curv":    pred_curv.numpy(),
        "step_size_z":  step_size_z.numpy(),
        # 1-D final-latent vectors [*]
        "norm_z0":      norm_z0.numpy(),
        "speed_z0":     speed_z0.numpy(),
        "curvature_z0": curv_z0.numpy(),
        "angular_z0":   ang_z0.numpy(),
        "pred_mag0":    pred_mag0.numpy(),
        # Dissolve scalars
        "dissolve_frame":      dissolve_frame,
        "dissolve_frame_free": dissolve_frame_free,
        "dissolve_strength":   dissolve_strength,
        "angular_min":         angular_min,
        "dissolve_by_step":    dissolve_by_step,
        "timesteps":           ts,
        "S": S, "F": F,
    }

print("Feature utilities ready.")

In [ ]:
# ── Plotting Utilities ────────────────────────────────────────────────────────

def _shade_cond(ax, data_len: int, F: int = F_PRIME, k: int = K_LAT, e: int = END_IDX,
                alpha: float = 0.13) -> None:
    """Add conditioning region shading and boundary dashes to an axis.

    data_len: number of data points along x (F, F-1, or F-2 depending on feature).
    The conditioning boundaries are in frame-space; for shorter features (speed/curv/ang),
    they shift by offset/2 because each column represents a window between frames.
    """
    offset = F - data_len        # 0 (norm/pred), 1 (speed), 2 (curv/ang)
    shift  = offset / 2.0
    start_bnd = (k - 0.5) - shift
    end_bnd   = (e - 0.5) - shift
    ax.axvspan(-0.5,       start_bnd,        alpha=alpha, color="#00BCD4", zorder=0)
    ax.axvspan(end_bnd,    data_len - 0.5,   alpha=alpha, color="#E91E63", zorder=0)
    if -0.5 < start_bnd < data_len - 0.5:
        ax.axvline(start_bnd, color="#00BCD4", lw=1.3, ls="--", alpha=0.9, zorder=1)
    if -0.5 < end_bnd   < data_len - 0.5:
        ax.axvline(end_bnd,   color="#E91E63", lw=1.3, ls="--", alpha=0.9, zorder=1)


def heatmap(ax, data: np.ndarray, title: str, cmap: str, diverging: bool = False,
            vmin=None, vmax=None, F: int = F_PRIME, k: int = K_LAT, e: int = END_IDX) -> None:
    """Draw one (S × F') heatmap with conditioning markers.  Row 0 = noisiest step."""
    S_dim, P_dim = data.shape
    if diverging:
        vmin, vmax = -1.0, 1.0
    else:
        vmin = vmin if vmin is not None else 0.0
        vmax = vmax if vmax is not None else float(np.percentile(data, 99))
    im = ax.imshow(data, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax,
                   origin="upper", interpolation="nearest",
                   extent=[-0.5, P_dim - 0.5, S_dim - 0.5, -0.5])
    ax.set_title(title, fontsize=7.5, pad=3)
    ax.set_xlabel("frame  p", fontsize=7)
    ax.set_ylabel("step τ  (0=noisy)", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.set_xticks(range(P_dim))
    # conditioning lines adjusted for feature offset
    offset = F - P_dim
    shift  = offset / 2.0
    for bnd, col in [(k - 0.5, "#00BCD4"), (e - 0.5, "#E91E63")]:
        bc = bnd - shift
        if -0.5 < bc < P_dim - 0.5:
            ax.axvline(bc, color=col, lw=1.2, ls="--", alpha=0.9)
    plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)


def dissolve_markers(ax, df_global: int, df_free: int) -> None:
    """Yellow solid = global pred, orange dashed = free-mid pred."""
    ax.axvline(df_global, color="#FFC107", lw=2.0, ls="-",  label=f"global p*={df_global}", zorder=5)
    if df_free != df_global:
        ax.axvline(df_free,   color="#FF6D00", lw=1.8, ls="--", label=f"free-mid p*={df_free}", zorder=5)


# ── Video Embedding ───────────────────────────────────────────────────────────

def video_html(path, width: int = 380, label: str = "", cls: str = "") -> str:
    """Return an HTML string embedding a video as base64."""
    if path is None or not pathlib.Path(path).exists():
        return f'<div style="width:{width}px;text-align:center;color:#aaa">[no video]</div>'
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    color = CLASS_COLORS.get(cls, "#888")
    lbl_html = (f'<p style="font-size:11px;font-weight:bold;margin:2px 0;'
                f'color:{color}">{label}</p>') if label else ""
    return (f'<div style="display:inline-block;text-align:center;padding:6px;'
            f'border:2px solid {color};border-radius:6px;margin:4px">'
            f'{lbl_html}'
            f'<video width="{width}" controls>'
            f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
            f'</video></div>')


def video_grid(records, cols: int = 5, width: int = 290) -> HTML:
    """Render all generated videos in a responsive grid."""
    cells = [
        f'<td style="padding:4px;vertical-align:top">'
        f'{video_html(r["video_path"], width=width, label=r["short_id"], cls=r["cls"])}</td>'
        for r in records
    ]
    rows = [
        "<tr>" + "".join(cells[i:i+cols]) + "</tr>"
        for i in range(0, len(cells), cols)
    ]
    legend = "".join(
        f'<span style="background:{v};color:white;padding:2px 7px;border-radius:3px;'
        f'font-size:11px;margin:2px">[C{k}] {CLASS_LABELS[k]}</span>'
        for k, v in CLASS_COLORS.items()
    )
    return HTML(
        f'<div style="margin-bottom:8px">{legend}</div>'
        f'<table style="border-collapse:collapse">{"".join(rows)}</table>'
    )

print("Plotting utilities ready.")

In [ ]:
# ── Load All Samples & Compute Features ──────────────────────────────────────
# Memory strategy:
#   • z_t / v_pred are each ~100 MB bfloat16 per sample.
#   • hidden_states are ~2.3 GB per sample (6 steps × 4 layers × [2,6144,4096]).
#   • We extract all needed features upfront then DISCARD the raw tensors.
#   • hidden_states are loaded lazily one-sample-at-a-time in Level 3 cells.
#   • Total RAM for all records: < 100 MB (numpy feature arrays only).

import gc

def load_record(sample_dir: pathlib.Path) -> dict | None:
    traj_files  = sorted(sample_dir.glob("*_trajectory_stage1.pt"))
    video_files = sorted(sample_dir.glob("*.mp4"))
    if not traj_files:
        return None

    traj_path = traj_files[0]
    data      = torch.load(traj_path, weights_only=False, map_location="cpu")
    sample_id = data.get("sample_id", sample_dir.name)
    cls       = sample_id.split("__")[0].replace("class", "").strip()
    short_id  = sample_id.replace(f"class{cls}__", "").replace("__", " → ")
    has_hs    = bool(data.get("hidden_states"))

    feats = compute_features(data)
    del data
    gc.collect()

    return {
        "sample_id":  sample_id,
        "cls":        cls,
        "short_id":   f"[C{cls}] {short_id}",
        "cls_label":  CLASS_LABELS.get(cls, "?"),
        "color":      CLASS_COLORS.get(cls, "#888"),
        "video_path": video_files[0] if video_files else None,
        "traj_path":  traj_path,
        "has_hs":     has_hs,
        "feats":      feats,
    }

sample_dirs = sorted(
    p for p in TRAJ_DIR.iterdir()
    if p.is_dir() and list(p.glob("*_trajectory_stage1.pt"))
)
records = [r for d in sample_dirs if (r := load_record(d)) is not None]
records.sort(key=lambda r: (r["cls"], r["sample_id"]))

print(f"\u2713 Loaded {len(records)} samples  (large tensors freed)")
print(f"{'sample_id':<50s} {'cls':<3s} {'strength':>9s} {'ang_min':>8s} {'dis_frame':>10s}")
print("\u2500" * 85)
for r in records:
    f = r["feats"]
    print(f"  {r['sample_id']:<48s}  {r['cls']:<3s}"
          f"  {f['dissolve_strength']:>8.2f}  {f['angular_min']:>8.3f}"
          f"  p={f['dissolve_frame_free']:>2d} ({f['dissolve_frame_free']*LTX_TEMPORAL_SCALE/VIDEO_FPS:.2f}s)")

# ── Ground Truth Transition Annotations ───────────────────────────────────────
GT_CSV = pathlib.Path(
    "/workspace/diffusion-research/outputs/analysis/"
    "exp_022_geometric_features/run_0004/dissolve_table.csv"
)
_gt_df  = pd.read_csv(GT_CSV)
gt_dict = {}   # sample_id → gt_s (float seconds) or None

for _, row in _gt_df.iterrows():
    sid = row["sample_id"]
    try:
        gs = float(row["gt_s"])
        gt_dict[sid] = None if (gs != gs) else gs   # NaN check
    except (ValueError, TypeError):
        gt_dict[sid] = None

def gt_latent(sample_id: str) -> float | None:
    """GT transition time to latent frame (float, sub-frame precision)."""
    gs = gt_dict.get(sample_id)
    return gs * VIDEO_FPS / LTX_TEMPORAL_SCALE if gs is not None else None

def _add_gt_vline(ax, sample_id: str, color: str = "#4CAF50",
                  lw: float = 2.0, ls: str = "-", label: bool = True) -> None:
    """Draw green GT vertical line on a 1-D frame axis (if annotation exists)."""
    gp = gt_latent(sample_id)
    if gp is not None:
        lbl = f"GT {gt_dict[sample_id]:.2f}s" if label else None
        ax.axvline(gp, color=color, lw=lw, ls=ls, alpha=0.9,
                   label=lbl, zorder=7)

n_gt = sum(1 for v in gt_dict.values() if v is not None)
print(f"\n\u2713 GT annotations: {n_gt}/{len(gt_dict)} samples")
for sid, gs in sorted(gt_dict.items()):
    gp_str = f"  p={gt_latent(sid):.1f}" if gs is not None else "  —"
    print(f"  {sid:<52s}  {str(gs) if gs is not None else '—':>7s}s{gp_str}")


---
## Video Gallery — All 10 Generated Clips

Generated clips sorted by semantic class (border colour = class colour).  
Play each video **before** looking at the latent analysis — note where you perceive the visual cut or dissolve,  
and compare that against the predictions in the heatmaps below.

**Semantic class key:**
- 🔵 Class 1: similar context / similar category / similar motion (easiest)
- 🟢 Class 2: similar context / similar category / different motion
- 🟠 Class 5: different context / similar category / similar motion
- 🔴 Class 6: different context / similar category / different motion
- 🟣 Class 8: different context / different category / different motion (hardest)

In [ ]:
display(video_grid(records, cols=5, width=300))

---
## Level 1 — VAE Latent Space (`z_t`)

`z_t[τ, :, p, :, :]` is the **noisy video latent** at denoising step `τ`, latent frame `p`.  
Shape per element: `[C=128, H'=16, W'=24]` — 128 channels, 16 height positions, 24 width positions.

- At **τ=0**: nearly pure Gaussian noise — no content structure yet.
- At **τ=39**: the fully denoised clean latent `z₀`, which decodes into the video you see.

---

### Features extracted (each aggregates over C=128 channels and H'×W'=384 spatial positions)

| Feature | Computation | What it measures |
|---------|-------------|-----------------|
| **`norm_z[τ, p]`** | `‖z_t[τ, :, p, :, :]‖₂` | Noise level proxy. High at τ=0 (all noise), decreasing toward τ=39. Conditioned frames should separate from free-middle early. |
| **`speed_z[τ, p]`** | `‖z_t[τ,:,p+1,:,:] − z_t[τ,:,p,:,:]‖₂` | Spatial displacement between adjacent latent frames at step τ. Large = large content difference between frame `p` and `p+1`. |
| **`curvature_z[τ, p]`** | `‖z_t(p+2) − 2z_t(p+1) + z_t(p)‖₂` | **Primary dissolve signal.** Measures how much the frame sequence "bends" at frame `p`. A spike = hard turn in latent space = transition there. |
| **`angular_z[τ, p]`** | `cos(Δz_t(p),  Δz_t(p+1))` | Direction consistency. cos < 0 = the latent trajectory **reverses direction** at frame `p` = hard cut. |
| **`step_size_z[τ, p]`** | `‖z_t[τ+1,p] − z_t[τ,p]‖₂` | How much frame `p` changes per denoising step. Frames that change a lot late in denoising are uncertain — the model hasn't committed to their content. |

The most diagnostic signals come from **`z₀ = z_t[τ=39]`** (the clean latent), since it directly describes the output video structure.

> **Shading in all Level 1 plots:** 🔵 Cyan = start-clip `p=0..3` · 🟣 Magenta = end-clip `p=12..15` · ⬜ White = free-middle `p=4..11`

In [ ]:
# ── Level 1 · VAE Latent Heatmaps — per sample (6-panel) ─────────────────────
# Each panel shows one feature as a (τ × p) heatmap.  Rows = denoising steps (0=noisy),
# columns = latent frames (0..15).

n = len(records)
fig = plt.figure(figsize=(18, n * 2.4 + 1.0))
fig.suptitle("Level 1 — VAE Latent Features  (τ × p)  |  cyan=start  magenta=end",
             fontsize=12, fontweight="bold", y=0.995)

panels = [
    ("norm_z",      "‖z_t(p)‖  noise level",    "PuBu_r",   False),
    ("speed_z",     "speed_z  ‖Δ_p z_t‖",        "YlOrRd",   False),
    ("curvature_z", "curvature_z  ‖Δ²_p z_t‖ ◄DISSOLVE", "hot", False),
    ("angular_z",   "angular_z  cos(Δ,Δ) ◄FLIP", "RdYlGn",   True),
    ("pred_mag",    "pred_mag  ‖v_θ(p)‖",         "plasma",   False),
    ("step_size_z", "step_size  ‖Δ_τ z_t(p)‖",   "magma",    False),
]

n_cols = len(panels)
gs = gridspec.GridSpec(n, n_cols, figure=fig, hspace=0.55, wspace=0.38)

for row, r in enumerate(records):
    f = r["feats"]
    F = f["F"]
    for col, (key, title, cmap, div) in enumerate(panels):
        ax = fig.add_subplot(gs[row, col])
        data_2d = f[key]
        heatmap(ax, data_2d, title if row == 0 else "", cmap, div, F=F)
        if col == 0:
            cls_lbl = r["short_id"]
            ax.set_ylabel(f"{cls_lbl}\nstep τ", fontsize=6.5, labelpad=3)

plt.savefig("/tmp/level1_heatmaps.png", dpi=100, bbox_inches="tight")
plt.show()
print("Tip: row = one sample;  column = one feature.  Compare rows across classes.")

### Reading the Level 1 Heatmap Grid

**10 rows × 6 columns.** Each cell is a 2D heatmap:
- **x-axis** = latent frame `p` (0–15, i.e. 0 s → 5.3 s)
- **y-axis** = denoising step `τ` (row 0 at top = noisy; row 39 at bottom = clean)
- **Colour** = feature value. One global colour scale per column, shared across all 10 rows.

**Column-by-column guide:**

| Column | What it shows | What to look for |
|--------|---------------|-----------------|
| `‖z_t(p)‖` | L2 norm of the latent slice at frame `p`, step `τ` | Uniform high norm at τ=0 (pure noise). Conditioned frames (cyan/magenta) should deviate from free-middle early in denoising. |
| `speed_z` | `‖z_t(p+1)−z_t(p)‖` — displacement between adjacent latent frames | Bright column at `p` = large content change there. Persistent across τ = the model always sees a big jump at that frame boundary. |
| `curvature_z ◄` | `‖Δ²_p z_t‖` — second-order bend in the frame sequence | **Primary signal.** A bright pixel at `(τ, p)` = the latent bends at frame `p` during step `τ`. A bright vertical stripe = committed dissolve. |
| `angular_z` | `cos(Δz_t(p), Δz_t(p+1))` — angle between consecutive frame-differences | Red (cos < 0) = direction reversal at frame `p`. A red column = persistent flip throughout denoising. |
| `pred_mag` | `‖v_θ(p)‖` — velocity norm | Persistent bright stripe at `p*` from early to late τ = the model is confidently pushing frame `p*` toward very different content across all 40 steps. |
| `step_size_z` | `‖z_t[τ+1,p]−z_t[τ,p]‖` — per-step change | Frames that stay bright until late τ = uncertain content. Conditioned frames should go dark early. |

**Cross-row comparison (key experiment):** Does class 8 (purple rows) show a sharper, more persistent `curvature_z` stripe than class 1 (blue rows)?

In [ ]:
# ── Level 1 · Per-sample deep dive: video + z_t norm/curvature/angular ────────
# All signals from the FINAL clean latent z₀.  Green solid line = GT annotation.

n = len(records)
fig, axes = plt.subplots(n, 4, figsize=(18, n * 2.0))
fig.suptitle(
    "Level 1 — Clean Latent z₀ Signals  |  cyan=start  magenta=end  "
    "|  🟡=pred p*  🟢=GT annotation",
    fontsize=11, fontweight="bold"
)

for row, r in enumerate(records):
    f      = r["feats"]
    F      = f["F"]
    frames = np.arange(F)
    f_c2   = np.arange(F - 2) + 1.0
    df     = f["dissolve_frame"]
    df_fr  = f["dissolve_frame_free"]
    col    = r["color"]
    sid    = r["sample_id"]

    for colidx in range(4):
        ax = axes[row, colidx]
        if colidx == 0:
            ax.bar(frames, f["norm_z0"], width=0.75, color=col, alpha=0.8)
            dissolve_markers(ax, df, df_fr)
            _add_gt_vline(ax, sid, label=(row == 0))
            _shade_cond(ax, F, F=F)
            ax.set_xlim(-0.5, F-0.5); ax.set_xticks(range(F))
            ax.tick_params(labelsize=6)
            if row == 0: ax.set_title("‖z₀(p)‖  latent energy", fontsize=8)
            ax.set_ylabel(r["short_id"], fontsize=6, labelpad=2)
        elif colidx == 1:
            ax.bar(f_c2, f["curvature_z0"], width=0.75, color="#FF5722", alpha=0.85)
            dissolve_markers(ax, df, df_fr)
            _add_gt_vline(ax, sid, label=(row == 0))
            _shade_cond(ax, F-2, F=F)
            ax.set_xlim(-0.5, F-0.5); ax.set_xticks(range(F))
            ax.tick_params(labelsize=6)
            str_lbl = f"strength={f['dissolve_strength']:.1f}×"
            if row == 0: ax.set_title(f"‖Δ²z₀‖  curvature  ◄ DISSOLVE\n{str_lbl}", fontsize=8)
            else: ax.set_title(str_lbl, fontsize=7)
        elif colidx == 2:
            ang = f["angular_z0"]
            ax.plot(f_c2, ang, "o-", color="#2196F3", lw=1.5, ms=3.5)
            ax.axhline(0, color="gray", lw=0.8, ls=":")
            ax.fill_between(f_c2, ang, 0, where=(ang < 0), color="red", alpha=0.3)
            dissolve_markers(ax, df, df_fr)
            _add_gt_vline(ax, sid, label=(row == 0))
            _shade_cond(ax, F-2, F=F)
            ax.set_ylim(-1.15, 1.15)
            ax.set_xlim(-0.5, F-0.5); ax.set_xticks(range(F))
            ax.tick_params(labelsize=6)
            min_lbl = f"min={f['angular_min']:.3f}"
            if row == 0: ax.set_title(f"cos(Δ,Δ)  direction consistency\n{min_lbl}", fontsize=8)
            else: ax.set_title(min_lbl, fontsize=7)
        else:
            ax.imshow(f["step_size_z"].T, aspect="auto", cmap="magma",
                      origin="lower", extent=[-0.5, f["S"]-0.5, -0.5, F-0.5])
            gp = gt_latent(sid)
            if gp is not None:
                ax.axhline(gp, color="#4CAF50", lw=1.5, ls="-", alpha=0.9)
            ax.set_xlabel("step τ", fontsize=6); ax.set_ylabel("frame p", fontsize=6)
            ax.tick_params(labelsize=6)
            if row == 0: ax.set_title("step_size_z  ‖Δ_τ z_t(p)‖\n(when does frame p settle?)", fontsize=8)
    if row == 0:
        axes[row, 0].legend(fontsize=6, loc="upper right")

plt.tight_layout()
plt.show()


### Reading the Clean Latent `z₀` Signal Panels

**One row per sample. Four panels per row. All signals are from `z₀ = z_t[τ=39]` — the fully denoised latent that decodes into the video you see.**

**Panel 1 — `‖z₀(p)‖` latent energy (bar chart)**  
x-axis = latent frame `p` (0–15). y-axis = L2 norm over 128 channels and 16×24 spatial positions.  
Measures how much "signal energy" each frame carries in the clean latent.

**Panel 2 — `‖Δ²z₀‖` curvature — PRIMARY DISSOLVE SIGNAL (bar chart)**  
Value at position `p` = `‖z₀(p+1) − 2z₀(p) + z₀(p-1)‖₂` (centred second-difference, valid for p=1..14).  
**A tall bar = the latent trajectory bends sharply at frame `p` = a transition/dissolve is here.**  
`strength = peak / mean`. Values > 2 = detectable spike. Values > 4 = hard cut.  
**Yellow dashed line** = `p*` (argmax of curvature restricted to free-middle region `p=4..11`).

**Panel 3 — `cos(Δ,Δ)` angular consistency (line plot)**  
Value at `p` = cosine between `z₀(p+1)−z₀(p)` and `z₀(p+2)−z₀(p+1)`.  
cos ≈ +1 → smooth consistent motion. cos < 0 (red fill) → **direction reversal = hard cut.**  
`angular_min` = most negative value for this sample.

**Panel 4 — `‖Δ_τ z_t(p)‖` step-size heatmap (2D image)**  
x-axis = denoising step τ (0→39). y-axis = latent frame `p` (0→15). Brightness = how much frame `p` changed between steps `τ` and `τ+1`.  
**Dark = settled** (model committed to content). **Bright late = uncertain** (model still deciding).  
Conditioned frames should go dark early; the transition frame `p*` may stay bright longer.

In [ ]:
# ── Level 1 · Per-sample: video + key 1-D z_0 signals side-by-side ────────────
import io

all_html = []
all_html.append(
    '<h3 style="font-family:sans-serif">Level 1 — Video + z\u2080 Signals Per Sample</h3>'
    '<p style="font-family:sans-serif;font-size:12px">Each row: '
    '\U0001F3AC video  |  curvature  |  angular consistency  |  velocity magnitude<br>'
    '\U0001F7E1 dashed = predicted p*  \U0001F7E2 solid = GT annotation</p>'
)

for r in records:
    f      = r["feats"]
    F      = f["F"]
    f_c2   = np.arange(F - 2) + 1.0
    frames = np.arange(F)
    df     = f["dissolve_frame_free"]
    ang    = f["angular_z0"]
    col    = r["color"]
    sid    = r["sample_id"]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 2.4))
    gs_val = gt_dict.get(sid)
    gt_str = f"  |  GT={gs_val:.2f}s" if gs_val is not None else ""
    fig.suptitle(r["short_id"] + f"  |  Class {r['cls']}: {r['cls_label']}" + gt_str,
                 fontsize=9, fontweight="bold", color=col)

    # curvature
    ax1.bar(f_c2, f["curvature_z0"], width=0.75, color="#FF5722", alpha=0.85)
    ax1.axvline(df, color="#FFC107", lw=2.2, ls="--", label=f"p*={df}")
    _add_gt_vline(ax1, sid, label=True)
    _shade_cond(ax1, F-2, F=F)
    ax1.set_xlim(-0.5, F-0.5); ax1.set_xticks(range(F))
    ax1.set_title(f"curvature_z0  [strength={f['dissolve_strength']:.1f}\u00d7]", fontsize=8)
    ax1.set_xlabel("frame p"); ax1.legend(fontsize=7)
    ax1.tick_params(labelsize=7)

    # angular
    ax2.plot(f_c2, ang, "o-", color="#2196F3", lw=1.5, ms=3.5)
    ax2.axhline(0, color="gray", lw=0.8, ls=":")
    ax2.fill_between(f_c2, ang, 0, where=(ang < 0), color="red", alpha=0.3)
    ax2.axvline(df, color="#FFC107", lw=2.0, ls="--")
    _add_gt_vline(ax2, sid, label=False)
    _shade_cond(ax2, F-2, F=F)
    ax2.set_ylim(-1.15, 1.15); ax2.set_xlim(-0.5, F-0.5); ax2.set_xticks(range(F))
    ax2.set_title(f"angular_z0  [min={f['angular_min']:.3f}]", fontsize=8)
    ax2.set_xlabel("frame p")
    ax2.tick_params(labelsize=7)

    # pred_mag0
    ax3.bar(frames, f["pred_mag0"], width=0.75, color="#9C27B0", alpha=0.85)
    ax3.axvline(df, color="#FFC107", lw=2.0, ls="--")
    _add_gt_vline(ax3, sid, label=False)
    _shade_cond(ax3, F, F=F)
    ax3.set_xlim(-0.5, F-0.5); ax3.set_xticks(range(F))
    ax3.set_title("pred_mag0  ‖v_\u03b8(p)‖  final step", fontsize=8)
    ax3.set_xlabel("frame p")
    ax3.tick_params(labelsize=7)

    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode()

    vid_html = video_html(r["video_path"], width=340, label=r["short_id"], cls=r["cls"])
    plot_html = (f'<img src="data:image/png;base64,{img_b64}" '
                 f'style="height:200px;vertical-align:top">')
    all_html.append(
        f'<div style="display:flex;align-items:flex-start;gap:10px;'
        f'margin-bottom:10px;border-bottom:1px solid #ddd;padding-bottom:8px">'
        f'{vid_html}{plot_html}</div>'
    )

display(HTML("\n".join(all_html)))


---
### Level 1 — PCA of VAE Latent Frame Embeddings (`z₀`)

A complement to the bar-chart signals above: project the 16 frame embeddings of the **clean VAE latent `z₀`** into 2D to see their geometric layout.

**Computation:** for each latent frame `p`, average `z₀[C=128, :, :]` over the 16×24 spatial positions → one 128-dim vector. Stack all 16 → matrix `[F'=16, C=128]` → centre → SVD → 2D scatter.

- **Axes:** PC1, PC2 (top two axes of variance across the 16 frame embeddings of `z₀`)
- **Colour:** 🔵 blue=frame 0 → 🔴 red=frame 15 (temporal order)
- **★** = conditioned frame (`p=0..3`, `p=12..15`) · **●** = free-middle (`p=4..11`)
- **🟡 ring** = predicted `p*` (curvature argmax) · **🟢 diamond** = GT annotation (if available)
- **EV** = explained variance of PC1 / PC2

**What to look for:** conditioned frames (★) should anchor opposite ends; free-middle (●) frames interpolate between them. A sharp kink in the trajectory = the VAE latent also shows a discontinuity at `p*`. Compare shape across classes — do class 8 samples show a more pronounced kink than class 1?

In [ ]:
# ── Level 1 · PCA of VAE Latent Frame Embeddings (z₀) — all 10 samples ──────
# Loads z_final lazily per sample (~3 MB each, no hidden states needed).
# Shows all samples in a grid for cross-class comparison.

cmap_frames = plt.get_cmap("RdYlBu_r")
n_cols = 5
n_rows = (len(records) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.6, n_rows * 3.6))
axes_flat = axes.flatten()

fig.suptitle(
    "Level 1 — PCA of VAE Latent Frame Embeddings  (z\u2080, final denoising step \u03c4=39)\n"
    "Each dot = latent frame p.  Blue=early \u2192 Red=late.  "
    "\u2605=conditioned  \u25cf=free-middle  \u25ef ring=pred p*  \u25c6=GT",
    fontsize=10, fontweight="bold"
)

for idx, r in enumerate(records):
    ax = axes_flat[idx]
    sid = r["sample_id"]
    df  = r["feats"]["dissolve_frame_free"]
    gp  = gt_latent(sid)   # float latent frame or None

    # Load z_final (only ~3 MB of actual data used, but full .pt must be opened)
    print(f"  VAE PCA {r['short_id']} ...", end=" ", flush=True)
    _data = torch.load(r["traj_path"], weights_only=False, map_location="cpu")
    z0    = _data["z_final"].float()    # [C=128, F', H', W']
    del _data; gc.collect()
    print("ok")

    # Frame mean-embeddings: mean over (H', W') → [F', C=128]
    frame_vecs = z0.mean(dim=(2, 3)).T.numpy()    # [16, 128]
    frame_vecs -= frame_vecs.mean(axis=0)
    U, S_sv, _ = np.linalg.svd(frame_vecs, full_matrices=False)
    coords  = U[:, :2] * S_sv[:2]                # [16, 2]
    expvar  = (S_sv[:2]**2) / ((S_sv**2).sum() + 1e-12)

    for p in range(F_PRIME):
        c_val  = cmap_frames(p / (F_PRIME - 1))
        marker = "*" if (p < K_LAT or p >= END_IDX) else "o"
        ms     = 130 if marker == "*" else 65
        ax.scatter(coords[p, 0], coords[p, 1], color=c_val, s=ms,
                   marker=marker, zorder=3, alpha=0.9)
        ax.annotate(str(p), (coords[p, 0], coords[p, 1]),
                    fontsize=5, ha="center", va="center",
                    color="white", fontweight="bold")
    for p in range(F_PRIME - 1):
        ax.annotate("", xy=(coords[p+1, 0], coords[p+1, 1]),
                    xytext=(coords[p, 0], coords[p, 1]),
                    arrowprops=dict(arrowstyle="-|>", color="gray", lw=0.6, alpha=0.4))

    # # Predicted p* — yellow ring
    # ax.scatter(coords[df, 0], coords[df, 1], s=300,
    #            facecolors="none", edgecolors="#FFC107", lw=2.5, zorder=5)

    # GT — green diamond (interpolated between two nearest integer frames)
    if gp is not None:
        p_lo = int(np.floor(gp)); p_hi = min(p_lo + 1, F_PRIME - 1)
        frac  = gp - p_lo
        gt_xy = coords[p_lo] * (1 - frac) + coords[p_hi] * frac
        ax.scatter(gt_xy[0], gt_xy[1], marker="D", s=110,
                   color="#4CAF50", zorder=6, edgecolors="white", lw=0.8)

    ax.set_title(
        f"{r['short_id']}\nEV {expvar[0]:.0%}/{expvar[1]:.0%}",
        fontsize=7, color=r["color"], fontweight="bold"
    )
    ax.set_xlabel("PC1", fontsize=6); ax.set_ylabel("PC2", fontsize=6)
    ax.tick_params(labelsize=5)

for ax in axes_flat[len(records):]:
    ax.set_visible(False)

sm = plt.cm.ScalarMappable(cmap=cmap_frames, norm=plt.Normalize(0, F_PRIME-1))
sm.set_array([])
fig.colorbar(sm, ax=axes_flat.tolist(), shrink=0.35, pad=0.01, label="frame p")
plt.tight_layout()
plt.show()
print("\u25ef yellow ring = predicted p*   \u25c6 green diamond = GT transition (if annotated)")


---
## Level 2 — Velocity Field (`v_pred`)

`v_pred[τ, :, p, :, :]` is the **flow-matching velocity prediction** at denoising step `τ`, latent frame `p`.  
Same shape as `z_t`: `[C=128, H'=16, W'=24]`.

In flow matching (LTX-2's training objective), the velocity `v_θ` estimates the direction from the current noisy state toward the clean target:

$$v_\theta(z_\tau, \tau, c) \approx z_0 - z_\tau \cdot (1 - \tau/T)^{-1}$$

The magnitude `‖v_pred[τ,:,p,:,:]‖` directly measures how hard the model is working to move frame `p` at step `τ`.

**Why this is the primary transition detector:**  
A frame undergoing a hard dissolve must travel a large distance in latent space (from noise toward content that is very different from its neighbours). Its velocity norm will be **persistently high across all 40 denoising steps**.  
Visualised as the `pred_mag` heatmap (τ × p), this appears as a **vertical bright stripe** at the transition frame `p*` — visible even from the earliest denoising steps.

In [ ]:
# ── Level 2 · v_pred magnitude heatmap (τ × p) — ALL samples ─────────────────

n     = len(records)
ncols = 5
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.5))
axes_flat = [ax for row in (axes if nrows > 1 else [axes])
             for ax in (row if hasattr(row, '__iter__') else [row])]
fig.suptitle(
    "Level 2 — pred_mag = ‖v_\u03b8(z_\u03c4, \u03c4, c)(p)‖\u2082  (\u03c4 \u00d7 p)\n"
    "Bright vertical stripe at p* = TRANSITION FRAME  |  "
    "\U0001F7E1 dashed=pred p*  \U0001F7E2 solid=GT  cyan=start  magenta=end",
    fontsize=11, fontweight="bold"
)

global_vmax = max(float(np.percentile(r["feats"]["pred_mag"], 99)) for r in records)

for i, (ax, r) in enumerate(zip(axes_flat, records)):
    f   = r["feats"]
    sid = r["sample_id"]
    im  = ax.imshow(f["pred_mag"], aspect="auto", cmap="plasma",
                    vmin=0, vmax=global_vmax, origin="upper", interpolation="nearest",
                    extent=[-0.5, f["F"]-0.5, f["S"]-0.5, -0.5])
    df  = f["dissolve_frame_free"]
    ax.axvline(df, color="#FFC107", lw=1.8, ls="--", label=f"p*={df}")
    ax.axvline(K_LAT  - 0.5, color="#00BCD4", lw=1.2, ls="--", alpha=0.8)
    ax.axvline(END_IDX - 0.5, color="#E91E63", lw=1.2, ls="--", alpha=0.8)
    # GT vertical line
    gp = gt_latent(sid)
    if gp is not None:
        ax.axvline(gp, color="#4CAF50", lw=2.0, ls="-",
                   label=f"GT {gt_dict[sid]:.2f}s", alpha=0.9)
    ax.set_title(f"{r['short_id']}", fontsize=7.5, color=r["color"])
    ax.set_xlabel("frame p", fontsize=7)
    ax.set_ylabel("step \u03c4", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.set_xticks(range(f["F"]))
    ax.legend(fontsize=5.5, loc="upper right", framealpha=0.7)
    plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)

for ax in axes_flat[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()
print("Compare: class 1/2 = diffuse/no stripe.  class 6/8 = sharp stripe.  GT \u2248 stripe \u2192 good predictor.")


### Reading the Velocity Magnitude Heatmaps (pred_mag)

**One heatmap per generated clip (10 total, 5 per row).**
- **x-axis** = latent frame `p` (0–15), each tick ≈ 0.33 s
- **y-axis** = denoising step `τ` (row 0 at top = noisy; row 39 at bottom = clean)
- **Colour** = `‖v_θ(z_τ, τ, c)(p)‖₂` — velocity norm at frame `p` and step `τ`, summed over 128 channels and 384 spatial positions. One global colour scale across all 10 heatmaps.

**Pattern guide:**

| Visual pattern | Meaning |
|----------------|---------|
| **Vertical bright stripe across all τ at frame `p*`** | Transition is here, committed from step τ=0. The model works hard on `p*` throughout all 40 steps. |
| **Bright stripe only in bottom rows (τ=25..39)** | Transition decided late — model was uncertain about its location until near the end of denoising. |
| **Uniformly bright across all frames** | No single transition frame — similar clips throughout (expected for class 1). |
| **Bright at magenta region `p=12..15`** | Expected artefact from end-clip conditioning. **Not a dissolve signal** — ignore the magenta region when searching for transitions. |

**Yellow dashed line** = `p*` from Level 1 curvature. Alignment with the bright stripe = Level 1 and Level 2 signals agree.

In [ ]:
# ── Level 2 · Cross-sample: curvature_z0 + angular_z0 comparison ─────────────

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=False)
fig.suptitle(
    "Level 2 — Cross-Sample Dissolve Signals (clean latent z\u2080)\n"
    "Dashed vertical = predicted p*  |  Solid vertical = GT annotation  |  colour = class",
    fontsize=11, fontweight="bold"
)

F    = records[0]["feats"]["F"]
f_c2 = np.arange(F - 2) + 1.0

for r in records:
    f    = r["feats"]
    col  = r["color"]
    lbl  = r["short_id"]
    df_f = f["dissolve_frame_free"]
    sid  = r["sample_id"]

    c0   = f["curvature_z0"]
    c0_n = c0 / (c0.max() + 1e-8)
    ax1.plot(f_c2, c0_n, "-", color=col, lw=2.0, alpha=0.85, label=lbl)
    ax1.axvline(df_f, color=col, lw=1.0, ls="--", alpha=0.65)
    _add_gt_vline(ax1, sid, color=col, lw=1.8, ls="-", label=False)

    ax2.plot(f_c2, f["angular_z0"], "-", color=col, lw=2.0, alpha=0.85, label=lbl)
    ax2.axvline(df_f, color=col, lw=1.0, ls="--", alpha=0.65)
    _add_gt_vline(ax2, sid, color=col, lw=1.8, ls="-", label=False)

ax1.set_title("Normalised curvature_z0 = ‖z\u2080(p+2)\u22122z\u2080(p+1)+z\u2080(p)‖\u2082 / max", fontsize=9)
ax1.set_ylabel("curvature / max", fontsize=9)
ax1.set_ylim(-0.05, 1.15)
ax1.legend(fontsize=7, ncol=2, loc="upper left", framealpha=0.8)
ax1.axhline(0, color="gray", lw=0.5, ls=":")

ax2.set_title("angular_z0 = cos(z\u2080(p+1)\u2212z\u2080(p),  z\u2080(p+2)\u2212z\u2080(p+1))", fontsize=9)
ax2.set_ylabel("cosine", fontsize=9)
ax2.set_ylim(-1.15, 1.15)
ax2.axhline(0, color="gray", lw=0.8, ls=":")
ax2.fill_between(f_c2, -1.15, 0, alpha=0.04, color="red", label="reversal zone")
ax2.legend(fontsize=7, ncol=2, loc="upper left", framealpha=0.8)

for ax in (ax1, ax2):
    _shade_cond(ax, F-2, F=F)
    ax.set_xticks(range(F))
    ax.set_xlim(-0.5, F-0.5)
    ax.set_xlabel("frame  p", fontsize=9)
    ax.tick_params(labelsize=8)
    ax2b = ax.twiny()
    ax2b.set_xlim(-0.5, F-0.5)
    ax2b.set_xticks(range(F))
    ax2b.set_xticklabels([f"{p*LTX_TEMPORAL_SCALE/VIDEO_FPS:.1f}s" for p in range(F)],
                         rotation=45, fontsize=5.5)
    ax2b.set_xlabel("video time", fontsize=7)

plt.tight_layout()
plt.show()


### Reading the Cross-Sample Curvature + Angular Comparison

**All 10 samples overlaid on the same axes. Colour = semantic class.**

**Top panel — normalised curvature `‖z₀(p+2)−2z₀(p+1)+z₀(p)‖₂ / max`:**  
Each curve = one sample's curvature profile over frames p=1..14, normalised to [0,1] for shape comparison.
- **Narrow peak** = hard cut at one specific frame
- **Broad hump** = gradual transition
- **Flat** = no transition
- Dashed verticals = each sample's predicted `p*`. Secondary top axis = video time in seconds.

**Bottom panel — angular consistency `cos(Δz₀(p), Δz₀(p+1))`:**  
Value at `p` = cosine between consecutive frame-difference vectors in `z₀`.  
cos = +1 → smooth. cos = −1 → full direction reversal = hard cut. **Red fill** (cos < 0) = reversal zone.

**What to compare across classes:**  
Do class 8/6 (purple/red) show sharper peaks and more negative angular values than class 1/2 (blue/green)?  
Do same-class samples cluster their `p*` predictions near similar frame positions?  
Outliers — a class-1 sample with a spike, or class-8 with a flat line — are the most informative cases.

In [ ]:
# ── Level 2 · Dissolve Frame Evolution over Denoising Steps ───────────────────

fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle(
    "Dissolve Frame Commitment — argmax(curvature_z[\u03c4]) vs. denoising step \u03c4\n"
    "When does the model \'decide\' where to place the transition?  "
    "Stable early = committed; jittery late = uncertain\n"
    "Dashed line = predicted final p*  |  Solid horizontal \u25b6 = GT latent frame",
    fontsize=10, fontweight="bold"
)

S     = records[0]["feats"]["S"]
steps = np.arange(S)

for r in records:
    f   = r["feats"]
    col = r["color"]
    lbl = r["short_id"]
    ax.plot(steps, f["dissolve_by_step"], "-", color=col, lw=2.0, alpha=0.85, label=lbl)
    # GT horizontal line
    gp = gt_latent(r["sample_id"])
    if gp is not None:
        ax.axhline(gp, color=col, lw=1.2, ls=":", alpha=0.7)

ax.axhspan(K_LAT, END_IDX - 1, alpha=0.07, color="gold",
           label=f"free-middle p=[{K_LAT}..{END_IDX-1}]")
ax.axhspan(0, K_LAT - 0.5, alpha=0.07, color="#00BCD4")
ax.axhspan(END_IDX - 0.5, F_PRIME, alpha=0.07, color="#E91E63")
ax.set_xlabel("denoising step \u03c4  (0=noisy, 39=clean)", fontsize=9)
ax.set_ylabel("estimated dissolve frame p*", fontsize=9)
ax.set_xticks(steps[::5])
ax.set_yticks(range(F_PRIME))

# Add secondary y-axis in seconds
ax2y = ax.twinx()
ax2y.set_ylim(ax.get_ylim())
tick_p = range(F_PRIME)
ax2y.set_yticks(list(tick_p))
ax2y.set_yticklabels([f"{p*LTX_TEMPORAL_SCALE/VIDEO_FPS:.1f}s" for p in tick_p], fontsize=7)
ax2y.set_ylabel("video time", fontsize=8)

ax.set_ylim(-0.5, F_PRIME - 0.5)
ax.tick_params(labelsize=8)
ax.legend(fontsize=7, ncol=2, loc="upper right", framealpha=0.85)
ax.grid(axis="both", linestyle=":", alpha=0.35)
ax.text(-1.5, K_LAT/2, "start\nclip", fontsize=7, ha="center", color="#00BCD4", fontweight="bold")
ax.text(-1.5, (K_LAT + END_IDX)/2, "free\nmiddle", fontsize=7, ha="center", color="#888")
ax.text(-1.5, (END_IDX + F_PRIME)/2, "end\nclip", fontsize=7, ha="center", color="#E91E63", fontweight="bold")
plt.tight_layout()
plt.show()
print("Dotted horizontals = GT latent frame per sample.  Convergence toward GT = model correctly committed.")


### Reading the Dissolve Frame Commitment Plot

**One line per sample (colour = semantic class). x-axis = denoising step `τ` (0=noisy, 39=clean). y-axis = estimated `p*` at that step = argmax of `curvature_z[τ, :]`.**

This tracks "which frame has the highest curvature at each point in the denoising process?"  
It answers: **when does the model decide where to place the transition?**

**Shaded bands:**
- **Gold** = valid free-middle region `p=4..11` (a real transition must land here)
- **Cyan** = start-clip `p=0..3` (curvature peak here = conditioning dominates, not a real dissolve)
- **Magenta** = end-clip `p=12..15` (same)

**Line pattern guide:**

| Pattern | Meaning |
|---------|---------|
| Flat horizontal in gold band from τ=0 | Model committed to transition location immediately — clear, confident conditioning |
| Starts in cyan/magenta, drops into gold band | Boundary frames dominate early; model gradually moves the peak to the free-middle |
| Jumpy across all 40 steps | No stable transition signal — curvature peak is noise |
| Enters gold band only after τ=25 | Ambiguous conditioning — very late commitment |

**`commit_step`** (summary table) = earliest τ where the line stays within ±1 frame of its final value. Lower = more decisive.

**For control experiments:** Latent-space intervention (`z_t` editing) must happen **before** the commit step — after that, the transition location is embedded in the latent geometry.

In [ ]:
# ── Level 2 · v_pred norm profile: early vs. late denoising ───────────────────

fig, axes = plt.subplots(len(records), 3, figsize=(16, len(records) * 1.9))
fig.suptitle(
    "pred_mag across denoising thirds  (early / middle / final)\n"
    "Per-frame velocity — persistent peak across thirds = early-committed transition\n"
    "\U0001F7E1 dashed=pred p*   \U0001F7E2 solid=GT",
    fontsize=10, fontweight="bold"
)

for row, r in enumerate(records):
    f      = r["feats"]
    pm     = f["pred_mag"]
    S      = f["S"]; F = f["F"]
    col    = r["color"]
    frames = np.arange(F)
    sid    = r["sample_id"]

    thirds = [(0, S//3, "early  \u03c4=0\u201313"),
              (S//3, 2*S//3, "middle \u03c4=14\u201327"),
              (2*S//3, S, "final  \u03c4=28\u201339")]
    for tcol, (t0, t1, tlbl) in enumerate(thirds):
        ax = axes[row, tcol] if len(records) > 1 else axes[tcol]
        mean_pm = pm[t0:t1].mean(axis=0)
        ax.bar(frames, mean_pm, width=0.75, color=col, alpha=0.8)
        ax.axvline(f["dissolve_frame_free"], color="#FFC107", lw=2.0, ls="--",
                   label=f"p*={f['dissolve_frame_free']}")
        _add_gt_vline(ax, sid, label=(tcol == 2 and row == 0))
        _shade_cond(ax, F, F=F)
        ax.set_xlim(-0.5, F-0.5); ax.set_xticks(range(F))
        ax.tick_params(labelsize=6)
        if row == 0: ax.set_title(tlbl, fontsize=8)
        if tcol == 0: ax.set_ylabel(r["short_id"], fontsize=6, labelpad=2)

plt.tight_layout()
plt.show()


### Reading the Denoising Thirds — Velocity Profiles

**Layout: 10 rows (samples) × 3 columns (thirds of the denoising process).**  
Each bar chart: x-axis = latent frame `p` (0–15). y-axis = `‖v_θ(p)‖` averaged over that denoising third.

| Column | Steps | What it captures |
|--------|-------|-----------------|
| **Early** (τ=0–13) | First 35% | Velocity in heavily noised space — reflects the model's prior about where large content changes are needed |
| **Middle** (τ=14–26) | Middle 32% | Structure forming — velocity stabilises |
| **Final** (τ=27–39) | Last 33% | Detail refinement — lower overall velocity |

**How to interpret:**
- **Yellow dashed line** (`p*`) aligns with the tallest bar in **all three columns** → transition committed from the very first step — strong, confident signal
- Peak only in the **final column** → model decided transition location late
- **Flat bars everywhere** → no transition needed (similar clips)

**Cross-sample:** Compare bar heights and peak positions across rows. Does class 8 consistently show a taller, more isolated peak than class 1?

---
## Level 3 — Transformer Hidden States (`hidden_states`)

This level inspects the **internal representations** inside LTX-2's transformer blocks during denoising. This is the deepest, most exploratory level — we look for geometric signals inside a 4096-dimensional activation space.

---

### What is a transformer block in LTX-2?

LTX-2 is a **Diffusion Transformer (DiT)** with **48 transformer blocks** (attention + FFN). We probe 4 depths:

| Label | Block index (of 48) | Depth | Expected role |
|-------|---------------------|-------|---------------|
| **L12** | Block 12 | 25% | Shallow — low-level spatial/positional patterns |
| **L24** | Block 24 | 50% | Mid — spatial + semantic mixing |
| **L35** | Block 35 | 73% | Deep — high-level semantic content |
| **L47** | Block 47 | 98% | Near-output — direct precursor to the velocity prediction |

---

### What is a token in this context?

LTX-2 uses **spatiotemporal packed sequences**: all spatial positions of all latent frames are flattened into one 1D token sequence that the transformer attends to jointly:

```
Sequence length = F' × H' × W' = 16 × 16 × 24 = 6,144 tokens
                  ↑               ↑       ↑
            latent frames    lat. height  lat. width
```

Token `(p × 16 × 24) + (h × 24) + w` corresponds to latent frame `p`, height `h`, width `w`.

---

### What the saved hidden states look like

```
hidden_states[τ][L].shape = [2, 6144, 4096]
                              ↑   ↑      ↑
                         CFG  tokens  hidden dim
```

- **`[0]`** = unconditional forward pass (no text prompt — used for CFG guidance scale)
- **`[1]`** = conditional forward pass (with text prompt) ← **always used here**
- **6,144** = 16×16×24 packed tokens
- **4,096** = LTX-2's transformer hidden dimension

To analyse **per-frame** behaviour, we unpack 6,144 tokens → `[F'=16, H'=16, W'=24, D=4096]`.

---

### What is a "frame mean-embedding"? (used in PCA and similarity)

For latent frame `p`, we have **16×24 = 384 token embeddings** of shape `[4096]`.  
We **average** those 384 vectors → one 4096-dim representative vector for the whole frame:

```
h̄^L(p)  =  mean over h=0..15, w=0..23  of  h^L(p, h, w)     shape: [4096]
```

Stack for all 16 frames → matrix `[16, 4096]`. These are the "frame mean-embeddings".

> This is: "if you had to summarise the entire content of latent frame `p` using one activation vector from block `L`, what would it be?"

---

### Probing schedule (6 denoising steps captured)

| Index | τ value (approx) | Stage |
|-------|-----------------|-------|
| Step 0 | τ ≈ 0 | Pure noise |
| Step 8 | τ ≈ 200 | Early denoising |
| Step 16 | τ ≈ 500 | Mid denoising ← used in PCA and similarity plots |
| Step 23 | τ ≈ 720 | Late-mid |
| Step 31 | τ ≈ 970 | Late |
| Step 39 | τ ≈ 1000 | Fully denoised `z₀` |

In [ ]:
# ── Level 3 · Helpers for hidden-state analysis ───────────────────────────────

LAYER_INDICES = [12, 24, 35, 47]    # hooked transformer block indices
STEP_INDICES  = [0, 8, 16, 23, 31, 39]

def unpack_hidden(h: torch.Tensor) -> torch.Tensor:
    """Unpack [2, N, D] hidden state to [2, F', H', W', D] spatial tensor."""
    B, N, D = h.shape
    assert N == F_PRIME * H_PRIME * W_PRIME, f"Unexpected N={N}"
    return h.reshape(B, F_PRIME, H_PRIME, W_PRIME, D)

def per_frame_norm(h_spatial: torch.Tensor, cond_idx: int = 1) -> np.ndarray:
    """Mean activation norm per latent frame.
    h_spatial: [2, F', H', W', D]
    Returns: [F'] (using the conditional batch element)
    """
    h_c = h_spatial[cond_idx]           # [F', H', W', D]
    norms = h_c.norm(dim=-1)             # [F', H', W']
    return norms.mean(dim=(1, 2)).numpy()   # [F'] mean over spatial

def pca_frame_embeddings(h_spatial: torch.Tensor, cond_idx: int = 1, n_comp: int = 2):
    """PCA over frame mean-embeddings.
    Returns: coords [F', n_comp], explained_var [n_comp]
    """
    h_c = h_spatial[cond_idx]                      # [F', H', W', D]
    # Mean over spatial positions → one vector per frame
    frame_vecs = h_c.mean(dim=(1, 2)).float().numpy()  # [F', D]
    # Centre
    frame_vecs = frame_vecs - frame_vecs.mean(axis=0)
    # SVD
    U, S, Vt = np.linalg.svd(frame_vecs, full_matrices=False)
    coords = U[:, :n_comp] * S[:n_comp]             # [F', n_comp]
    total_var = (S**2).sum()
    explained = (S[:n_comp]**2) / (total_var + 1e-12)
    return coords, explained

def cosine_sim_matrix(h_spatial: torch.Tensor, cond_idx: int = 1) -> np.ndarray:
    """[F' × F'] cosine similarity matrix between mean frame embeddings."""
    h_c = h_spatial[cond_idx]
    fv  = h_c.mean(dim=(1, 2)).float()    # [F', D]
    fv  = fv / fv.norm(dim=1, keepdim=True).clamp(min=1e-8)
    return (fv @ fv.T).numpy()            # [F', F']

print("Hidden-state analysis utilities ready.")
print(f"Layers to inspect: {LAYER_INDICES}")
print(f"Steps to inspect:  {STEP_INDICES}")

In [ ]:
# ── Level 3 · Per-frame activation norm  (layer × step heatmap) ───────────────
# 4 rows (transformer block depth) × 6 cols (denoising step) per sample.
# GT green vertical line added to each mini-chart.

n_samples_with_hs = sum(1 for r in records if r["has_hs"])
print(f"{n_samples_with_hs}/{len(records)} samples have hidden states.")
print("Loading one sample at a time (~2.4 GB) — may take 1-2 min total.\n")

if n_samples_with_hs == 0:
    print("No hidden states found.")
else:
    frames = np.arange(F_PRIME)

    for r in [rec for rec in records if rec["has_hs"]]:
        print(f"  Loading {r['sample_id']} ...", end=" ", flush=True)
        full_data = torch.load(r["traj_path"], weights_only=False, map_location="cpu")
        hs  = full_data["hidden_states"]
        df  = r["feats"]["dissolve_frame_free"]
        gp  = gt_latent(r["sample_id"])
        col = r["color"]
        sid = r["sample_id"]
        print("done")

        n_layers = len(LAYER_INDICES); n_steps = len(STEP_INDICES)
        fig, axes = plt.subplots(n_layers, n_steps, figsize=(n_steps*2.2, n_layers*2.0))
        gs_val = gt_dict.get(sid)
        gt_str = f"  |  GT {gs_val:.2f}s" if gs_val is not None else ""
        fig.suptitle(
            f"Level 3 — Per-frame hidden-state norm  ‖h^l(p)‖\n"
            f"{r['short_id']}{gt_str}  |  "
            f"\U0001F7E1 dashed=pred p*={df}  \U0001F7E2 solid=GT",
            fontsize=9, fontweight="bold", color=col
        )

        all_norms = []
        for step_idx in STEP_INDICES:
            for layer_idx in LAYER_INDICES:
                h_t = hs.get(step_idx, {}).get(layer_idx)
                if h_t is not None:
                    all_norms.append(per_frame_norm(unpack_hidden(h_t.float())))
        vmax_hs = max(n.max() for n in all_norms) if all_norms else 1.0

        for si, step_idx in enumerate(STEP_INDICES):
            for li, layer_idx in enumerate(LAYER_INDICES):
                ax   = axes[li, si]
                h_t  = hs.get(step_idx, {}).get(layer_idx)
                if h_t is None:
                    ax.set_visible(False); continue
                norms = per_frame_norm(unpack_hidden(h_t.float()))
                ax.bar(frames, norms, width=0.75, color=col, alpha=0.8)
                ax.axvline(df, color="#FFC107", lw=1.5, ls="--")
                if gp is not None:
                    ax.axvline(gp, color="#4CAF50", lw=1.5, ls="-", alpha=0.85)
                _shade_cond(ax, F_PRIME)
                ax.set_xlim(-0.5, F_PRIME-0.5); ax.set_xticks(range(0, F_PRIME, 4))
                ax.tick_params(labelsize=5.5); ax.set_ylim(0, vmax_hs*1.05)
                if li == 0:
                    lbl = f"\u03c4={step_idx}"
                    lbl += " (noisy)" if step_idx == 0 else (" (clean)" if step_idx == 39 else "")
                    ax.set_title(lbl, fontsize=7)
                if si == 0:
                    ax.set_ylabel(f"L{layer_idx}\n‖h‖", fontsize=7)

        plt.tight_layout()
        plt.show()
        del full_data, hs, all_norms
        gc.collect()
        print(f"  Memory freed.\n")


### Reading the Per-Frame Activation Norm Plots

**One figure per sample. Each figure: 4 rows (transformer block depth) × 6 columns (denoising step).**

**Each individual bar chart:**
- **x-axis** = latent frame `p` (0–15)
- **y-axis** = `‖h^L(p)‖` — the **mean L2 norm** of all 16×24=384 token embeddings in latent frame `p`, at transformer block `L` and denoising step `τ`:
  ```
  mean over h,w  of  ‖h^L(p, h, w)‖₂     (where each vector has D=4096 dimensions)
  ```
  This answers: "how much activation energy does each latent frame carry at this block and step?"

**Layout:**
- **Rows top→bottom** = transformer block depth: L12 (shallow) → L47 (deep, near-output)
- **Columns left→right** = denoising step: τ=0 (noisy, leftmost) → τ=39 (clean, rightmost)
- All bar charts share the same y-axis scale (comparable across the entire grid)

**What to look for:**
- **Conditioned frames** (cyan p=0..3, magenta p=12..15): should show elevated norms because the model has a precise conditioning target for these frames
- **Transition frame `p*`** (yellow dashed line): elevated norm at `p*` in deep blocks (L35/L47) and late steps (τ=31, 39) → the transformer allocates extra representational energy to the dissolve frame
- **Evolution left→right per row**: norms evolving as denoising progresses; a frame whose norm changes substantially is being actively constructed

**Null result:** Roughly uniform bars across all frames → the transformer distributes activation energy equally and does not explicitly mark transition frames.

### Level 3 — PCA of Transformer Frame Embeddings: All Samples × Multiple Timesteps

Previously only 4 samples at τ=16 were shown. Here we show **all samples** with a full grid:
- **Rows** = denoising step τ ∈ {0 (pure noise), 8 (early), 16 (mid), 39 (clean)}
- **Columns** = transformer block L ∈ {L12, L24, L35, L47} (shallow → deep)

Each dot = one latent frame `p`, coloured by temporal order (🔵 early → 🔴 late).  
**★** = conditioned · **●** = free-middle · **🟡 ring** = predicted `p*` · **🟢 diamond** = GT.

**Reading across rows (timesteps):** Does temporal structure emerge over denoising? At τ=0 (noise) expect no structure; by τ=39 (clean) expect clear ordering.  
**Reading across columns (blocks):** Do deeper blocks (L35, L47) show tighter semantic clusters than shallow (L12)?  
**Memory note:** each `.pt` file is ~2.4 GB; loaded and freed one sample at a time.

In [ ]:
# ── Level 3 · PCA — ALL samples, 4 denoising steps × 4 transformer blocks ────
# Rows = timestep (noise → clean)   Cols = block depth (shallow → deep)
# One figure per sample, loaded and freed individually.

PCA_TIMESTEPS = [0, 8, 16, 39]   # subset of STEP_INDICES=[0,8,16,23,31,39]
cmap_frames   = plt.get_cmap("RdYlBu_r")

hs_records = [r for r in records if r["has_hs"]]
if not hs_records:
    print("No hidden states — skipping PCA.")
else:
    print(f"PCA: {len(hs_records)} samples  ×  {len(PCA_TIMESTEPS)} timesteps  "
          f"×  {len(LAYER_INDICES)} blocks\n")

    for r in hs_records:
        sid = r["sample_id"]
        print(f"  Loading {sid} ...", end=" ", flush=True)
        full_data = torch.load(r["traj_path"], weights_only=False, map_location="cpu")
        hs  = full_data["hidden_states"]
        df  = r["feats"]["dissolve_frame_free"]
        gp  = gt_latent(sid)
        col = r["color"]
        print("done")

        n_rows = len(PCA_TIMESTEPS)
        n_cols = len(LAYER_INDICES)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.3, n_rows * 3.3))

        gs_val  = gt_dict.get(sid)
        gt_str  = f"  |  GT={gs_val:.2f}s (p\u2248{gp:.1f})" if gs_val is not None else ""
        fig.suptitle(
            f"Level 3 — PCA frame embeddings  ·  {r['short_id']}\n"
            f"Rows=denoising step \u03c4  |  Cols=transformer block L (depth)\n"
            f"Pred p*={df}{gt_str}  |  \u25c9=conditioned \u25cb=free-middle "
            f"\u25ef=p* \u25c6=GT",
            fontsize=9, fontweight="bold", color=col
        )

        for ri, step_idx in enumerate(PCA_TIMESTEPS):
            for ci, layer_idx in enumerate(LAYER_INDICES):
                ax  = axes[ri, ci]
                h_t = hs.get(step_idx, {}).get(layer_idx)
                if h_t is None:
                    ax.set_visible(False); continue

                coords, expvar = pca_frame_embeddings(unpack_hidden(h_t.float()))

                # Draw arrows first (under dots)
                for p in range(F_PRIME - 1):
                    ax.annotate("", xy=(coords[p+1, 0], coords[p+1, 1]),
                                xytext=(coords[p, 0], coords[p, 1]),
                                arrowprops=dict(arrowstyle="-|>", color="gray",
                                               lw=0.5, alpha=0.35))
                # Dots
                for p in range(F_PRIME):
                    c_val  = cmap_frames(p / (F_PRIME - 1))
                    marker = "*" if (p < K_LAT or p >= END_IDX) else "o"
                    ms     = 100 if marker == "*" else 50
                    ax.scatter(coords[p, 0], coords[p, 1], color=c_val, s=ms,
                               marker=marker, zorder=3, alpha=0.9)
                    ax.annotate(str(p), (coords[p, 0], coords[p, 1]),
                                fontsize=4.5, ha="center", va="center",
                                color="white", fontweight="bold")

                # Predicted p*: yellow ring
                ax.scatter(coords[df, 0], coords[df, 1], s=220,
                           facecolors="none", edgecolors="#FFC107", lw=2.2, zorder=5)

                # GT: green diamond (interpolated)
                if gp is not None:
                    p_lo = int(np.floor(gp)); p_hi = min(p_lo + 1, F_PRIME - 1)
                    frac = gp - p_lo
                    gt_xy = coords[p_lo] * (1 - frac) + coords[p_hi] * frac
                    ax.scatter(gt_xy[0], gt_xy[1], marker="D", s=80,
                               color="#4CAF50", zorder=6, edgecolors="white", lw=0.8)

                ax.tick_params(labelsize=5)
                ax.set_xlabel("PC1", fontsize=6)
                if ri == 0:
                    ax.set_title(f"L{layer_idx}  ({layer_idx/48:.0%})\nEV {expvar[0]:.0%}/{expvar[1]:.0%}",
                                 fontsize=7)
                else:
                    ax.set_title(f"EV {expvar[0]:.0%}/{expvar[1]:.0%}", fontsize=6)
                if ci == 0:
                    step_lbl = {0: "noisy", 8: "early", 16: "mid", 39: "clean"}.get(step_idx, "")
                    ax.set_ylabel(f"\u03c4={step_idx}\n({step_lbl})\nPC2", fontsize=6.5)

        sm = plt.cm.ScalarMappable(cmap=cmap_frames, norm=plt.Normalize(0, F_PRIME-1))
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.25, pad=0.01)
        cbar.set_label("frame p", fontsize=7)
        cbar.set_ticks([0, 4, 8, 12, 15])
        plt.tight_layout()
        plt.show()

        del full_data, hs
        gc.collect()
        print(f"  Memory freed.\n")

print("\u2605=conditioned  \u25cb=free-middle  \u25ef ring=pred p*  \u25c6 diamond=GT")


### Reading the Transformer PCA Grid

**One figure per sample. Grid: 4 rows (denoising step τ) × 4 cols (transformer block L).**

Each scatter plot = 16 frame mean-embeddings projected to 2D via SVD.  
Blue=frame 0 → Red=frame 15.  **★**=conditioned · **●**=free-middle · **🟡 ring**=pred p* · **🟢 diamond**=GT.

**Read across rows (τ: noise → clean):**  
- τ=0 (noise): usually random scatter — no content encoded yet.  
- τ=8 (early): coarse structure may begin to appear, driven by conditioning anchors.  
- τ=16 (mid): semantic content partially formed.  
- τ=39 (clean): clearest structure. Conditioned ★ frames anchor opposite ends; free-middle ● frames span between.

**Read across columns (L12 → L47):**  
- L12 (25% depth): mainly positional/spatial information from RoPE.  
- L47 (98% depth): semantic content dominates; expect tightest clustering.

**What to look for:**  
- **Kink in the frame trajectory** between p and p+1: does it align with pred p* (🟡) or GT (🟢)?  
- **Two clusters** (★ start group vs ★ end group with ● frames forced to interpolate): indicates the model has cleanly separated the two semantic halves.  
- **Collapse to one cluster**: this block/timestep has not yet separated the two clip semantics.

In [ ]:
# ── Level 3 · Frame cosine-similarity matrix across layers ────────────────────
# All samples. GT shown as green dashed lines crossing the matrix at gt_p.

SIM_TIMESTEPS = [8, 16, 39]   # show early / mid / clean
SIM_N_SHOW    = len([r for r in records if r["has_hs"]])   # all samples

hs_records = [r for r in records if r["has_hs"]]
if not hs_records:
    print("No hidden states — skipping similarity matrix.")
else:
    for r in hs_records:
        sid = r["sample_id"]
        print(f"Sim matrix: loading {sid} ...", end=" ", flush=True)
        full_data = torch.load(r["traj_path"], weights_only=False, map_location="cpu")
        hs  = full_data["hidden_states"]
        df  = r["feats"]["dissolve_frame_free"]
        gp  = gt_latent(sid)
        col = r["color"]
        print("done")

        for target_step in SIM_TIMESTEPS:
            step_lbl = {8: "early (\u03c4=8)", 16: "mid (\u03c4=16)", 39: "clean (\u03c4=39)"}[target_step]
            fig, axes = plt.subplots(1, len(LAYER_INDICES),
                                     figsize=(len(LAYER_INDICES) * 2.9, 3.0))
            gs_val = gt_dict.get(sid)
            gt_str = f"  GT {gs_val:.2f}s" if gs_val is not None else ""
            fig.suptitle(
                f"Level 3 — Frame cosine-sim  at {step_lbl}  ·  {r['short_id']}\n"
                f"\U0001F7E1 dashed=pred p*={df}  \U0001F7E2 dashed=GT{gt_str}  "
                f"Block-diagonal = hard semantic cut",
                fontsize=8.5, fontweight="bold", color=col
            )

            for ci, layer_idx in enumerate(LAYER_INDICES):
                ax  = axes[ci]
                h_t = hs.get(target_step, {}).get(layer_idx)
                if h_t is None:
                    ax.set_visible(False); continue

                sim = cosine_sim_matrix(unpack_hidden(h_t.float()))
                im  = ax.imshow(sim, vmin=-0.3, vmax=1.0, cmap="RdYlGn",
                                origin="upper", interpolation="nearest")
                ax.axhline(df - 0.5, color="#FFC107", lw=1.8, ls="--")
                ax.axvline(df - 0.5, color="#FFC107", lw=1.8, ls="--")
                for bnd, c in [(K_LAT - 0.5, "#00BCD4"), (END_IDX - 0.5, "#E91E63")]:
                    ax.axhline(bnd, color=c, lw=1.0, ls=":", alpha=0.8)
                    ax.axvline(bnd, color=c, lw=1.0, ls=":", alpha=0.8)
                # GT lines
                if gp is not None:
                    ax.axhline(gp - 0.5, color="#4CAF50", lw=1.8, ls="--", alpha=0.9)
                    ax.axvline(gp - 0.5, color="#4CAF50", lw=1.8, ls="--", alpha=0.9)
                ax.set_xticks(range(0, F_PRIME, 4)); ax.set_yticks(range(0, F_PRIME, 4))
                ax.tick_params(labelsize=6.5)
                ax.set_title(f"L{layer_idx}", fontsize=9)
                ax.set_xlabel("frame p", fontsize=7)
                if ci == 0:
                    ax.set_ylabel("frame q", fontsize=7)
                plt.colorbar(im, ax=ax, shrink=0.9, pad=0.02)

            plt.tight_layout()
            plt.show()

        del full_data, hs
        gc.collect()
        print(f"  Memory freed.\n")

print("\U0001F7E1 dashed=pred p*  \U0001F7E2 dashed=GT  Block diagonal in L35/L47 = hard semantic cut.")


### Reading the Frame Cosine-Similarity Matrices

**One row per sample (4 samples shown). One heatmap per transformer block (L12→L47), at denoising step τ=16.**

**Each [16×16] heatmap:**  
Entry `(p, q)` = cosine similarity between frame mean-embeddings `h̄^L(p)` and `h̄^L(q)`:
```
cos(h̄^L(p), h̄^L(q))    where  h̄^L(p) = mean of 384 token embeddings in latent frame p
```
Green = similar representations. Red = dissimilar.  
**Axes:** x-axis = latent frame `p` (0–15). y-axis = latent frame `q` (0–15).

**How to interpret:**

| Pattern | Meaning |
|---------|---------|
| **Block diagonal** (top-left + bottom-right green blobs, off-diagonal red) | Two distinct semantic clusters: frames 0..X ≈ each other, frames X+1..15 ≈ each other, but the two groups are very different. **The cluster boundary ≈ the dissolve frame.** |
| Smooth gradient (sim decreases with temporal distance) | Gradual content change — no hard cut |
| Uniform green everywhere | This block doesn't differentiate frames temporally — encodes spatial content only |

**Yellow dashed lines** = predicted `p*` from Level 1. Does the block-diagonal boundary in deep layers (L35/L47) align with `p*`? Alignment = Level 1 and Level 3 signals are consistent.  
**Cyan/magenta dotted lines** = conditioning boundaries (p=3.5 and p=11.5).

**Comparing L12 → L47:** Expect the block-diagonal to become sharper in deeper layers as representations become more semantically discriminative.

---
## Summary Table — Dissolve Metrics Across All Samples

All key per-sample scalars from Levels 1 and 2 in one styled table.

### Column definitions

| Column | Source | What it measures | How to read it |
|--------|--------|-----------------|----------------|
| `dis_p_free` | Level 1 — curvature peak of `z₀` | Predicted transition latent frame `p*`, restricted to free-middle `p=4..11` | Lower = transition earlier; higher = later in the video |
| `dis_t_s` | Converted from `dis_p_free` | Predicted transition time in seconds (`p* × 8 / 24`) | Compare against what you see when playing the video |
| `dis_p_global` | Level 1 — curvature over all 16 frames | Same but not restricted to free-middle | Should match `dis_p_free`; deviations = conditioning region dominates |
| `strength` | Level 1 | `max(curvature_z₀) / mean(curvature_z₀)` | 1.0 = flat (no spike). >2 = detectable cut. >4 = hard cut. |
| `angular_min` | Level 1 | Minimum cosine between consecutive frame-differences in `z₀` | 0 = orthogonal. Negative = direction reversal. More negative = harder cut. |
| `pred_peak_p` | Level 2 | Frame with highest mean velocity across all 40 denoising steps | Independent confirmation of transition location from velocity field |
| `commit_step` | Level 2 | Earliest τ where `argmax(curvature_z[τ])` stays within ±1 frame of final value | Lower = more decisive. Higher = model was uncertain. |

**Cell colours:** Strength = white→orange-red (higher = sharper). Angular_min = red→green (more negative = harder). Commit_step = light→dark blue (later = less decisive).

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────

def _commit_step(dissolve_by_step: np.ndarray, tolerance: int = 1) -> int:
    final = dissolve_by_step[-1]
    for i in range(len(dissolve_by_step)):
        if np.all(np.abs(dissolve_by_step[i:] - final) <= tolerance):
            return int(i)
    return len(dissolve_by_step) - 1

rows = []
for r in records:
    f         = r["feats"]
    pm        = f["pred_mag"]
    mean_pm   = pm.mean(axis=0)
    pred_peak = int(np.argmax(mean_pm))
    commit    = _commit_step(f["dissolve_by_step"])
    gs        = gt_dict.get(r["sample_id"])
    gp_approx = round(gt_latent(r["sample_id"]), 1) if gs is not None else None
    pred_t    = round(f["dissolve_frame_free"] * LTX_TEMPORAL_SCALE / VIDEO_FPS, 2)
    err_free  = round(abs(pred_t - gs), 2) if gs is not None else None

    rows.append({
        "sample":       r["short_id"],
        "class":        r["cls"],
        "cls_label":    r["cls_label"],
        "dis_p_free":   f["dissolve_frame_free"],
        "dis_t_s":      pred_t,
        "gt_s":         round(gs, 2) if gs is not None else None,
        "gt_p":         gp_approx,
        "err_s":        err_free,
        "strength":     round(f["dissolve_strength"], 2),
        "angular_min":  round(f["angular_min"], 3),
        "pred_peak_p":  pred_peak,
        "commit_step":  commit,
    })

df_table = pd.DataFrame(rows).sort_values(["class", "sample"]).reset_index(drop=True)

def _cls_color(cls_str):
    return CLASS_COLORS.get(str(cls_str), "#888")

def _row_style(row):
    col = _cls_color(str(row["class"]))
    return [f"background-color:{col}18; border-left:4px solid {col}"] * len(row)

styled = (
    df_table.style
    .apply(_row_style, axis=1)
    .background_gradient(subset=["strength"],    cmap="YlOrRd", vmin=1, vmax=5)
    .background_gradient(subset=["angular_min"], cmap="RdYlGn", vmin=-1, vmax=1)
    .background_gradient(subset=["commit_step"], cmap="PuBu",   vmin=0, vmax=39)
    .background_gradient(subset=pd.IndexSlice[df_table["err_s"].notna(), "err_s"],
                         cmap="YlOrRd", vmin=0, vmax=2)
    .format({
        "strength":    "{:.2f}×",
        "angular_min": "{:.3f}",
        "dis_t_s":     "{:.2f}s",
        "gt_s":        lambda x: f"{x:.2f}s" if x is not None else "—",
        "err_s":       lambda x: f"{x:.2f}s" if x is not None else "—",
    })
    .set_caption("Dissolve Metrics — all 10 samples  |  gt_s = manual GT  |  err_s = |pred − GT|")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold"),
                                          ("text-align", "left"), ("margin-bottom", "8px")]},
        {"selector": "th", "props": [("background-color", "#2c2c2c"), ("color", "white"),
                                     ("font-size", "11px"), ("padding", "5px 10px")]},
        {"selector": "td", "props": [("font-size", "11px"), ("padding", "4px 10px")]},
    ])
)
display(styled)

# Class-level aggregation
agg = df_table.groupby("class").agg(
    n=("sample", "count"),
    mean_strength=("strength", "mean"),
    mean_angular_min=("angular_min", "mean"),
    mean_commit_step=("commit_step", "mean"),
    mean_dis_t_s=("dis_t_s", "mean"),
    mean_err_s=("err_s", "mean"),
).round(3)
print("\nClass-level aggregation:")
display(agg)


### Reading the Summary Scatter and Bar Charts

**Left — Dissolve Strength vs. Direction Reversal (scatter)**  
x-axis = `strength` (max/mean curvature). y-axis = `angular_min` (most negative cosine).
- **Upper-left region** (high strength + negative angular_min) = hard, abrupt dissolve → expected for class 6/8
- **Lower-right region** (low strength + near-zero angular_min) = smooth or absent transition → expected for class 1

**Middle — Commitment Step by Sample (bar chart)**  
y-axis = denoising step τ at which the model's dissolve frame estimate stabilises.
- **Low bar** (τ < 10) = decided early — confident, clear conditioning
- **High bar** (τ > 30) = decided late — ambiguous conditioning

**Right — Predicted Transition Time by Class (bar + scatter)**  
x-axis = semantic class. y-axis = predicted transition time in seconds.  
Bar = class mean ± std. White dots = individual samples.  
All transitions should fall in the free-middle window: 1.3 s – 3.7 s (p=4..11).  
Does class difficulty correlate with transition timing?

In [ ]:
# ── Summary scatter: dissolve strength vs. angular_min coloured by class ──────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Summary — Geometric Dissolve Signals by Semantic Class  |  \U0001F7E2=GT available",
             fontsize=11, fontweight="bold")

# ── scatter: strength vs angular_min ─────────────────────────────────────────
ax = axes[0]
for r in records:
    f   = r["feats"]
    col = r["color"]
    sid = r["sample_id"]
    ax.scatter(f["dissolve_strength"], f["angular_min"], color=col, s=110, zorder=3,
               alpha=0.9, label=r["short_id"])
    ax.annotate(f"C{r['cls']}", (f["dissolve_strength"], f["angular_min"]),
                fontsize=8, ha="center", va="center", color="white", fontweight="bold")
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.axvline(1, color="gray", lw=0.8, ls=":")
ax.set_xlabel("dissolve strength  (max/mean curvature)", fontsize=9)
ax.set_ylabel("angular_min  (min cosine)", fontsize=9)
ax.set_title("Dissolve Strength vs. Direction Reversal", fontsize=9)
ax.legend(fontsize=6, ncol=1, loc="upper left", framealpha=0.8)

# ── bar: commit_step by sample ────────────────────────────────────────────────
ax = axes[1]
commit_vals = [_commit_step(r["feats"]["dissolve_by_step"]) for r in records]
bars = ax.bar(range(len(records)), commit_vals,
              color=[r["color"] for r in records], alpha=0.85)
ax.set_xticks(range(len(records)))
ax.set_xticklabels(
    [r["short_id"].replace("[C", "C").split("]")[0] + "]" for r in records],
    rotation=45, ha="right", fontsize=7
)
ax.set_ylabel("commit step \u03c4", fontsize=9)
ax.set_title("Denoising Step of Transition Commitment", fontsize=9)
ax.set_ylim(0, S_STEPS)
ax.axhline(S_STEPS/2, color="gray", lw=0.8, ls=":")

# ── scatter: predicted time vs GT time ───────────────────────────────────────
ax = axes[2]
for r in records:
    f   = r["feats"]
    col = r["color"]
    sid = r["sample_id"]
    pred_t = f["dissolve_frame_free"] * LTX_TEMPORAL_SCALE / VIDEO_FPS
    gs     = gt_dict.get(sid)
    if gs is not None:
        ax.scatter(gs, pred_t, color=col, s=110, zorder=3, alpha=0.9)
        ax.annotate(f"C{r['cls']}", (gs, pred_t),
                    fontsize=7, ha="center", va="center", color="white", fontweight="bold")

total_dur = F_PRIME * LTX_TEMPORAL_SCALE / VIDEO_FPS
ax.plot([0, total_dur], [0, total_dur], "k--", lw=0.9, alpha=0.5, label="perfect")
ax.set_xlim(0, total_dur); ax.set_ylim(0, total_dur)
ax.set_xlabel("GT transition time (s)", fontsize=9)
ax.set_ylabel("Predicted transition time (s)", fontsize=9)
ax.set_title("Predicted vs. GT Transition Time", fontsize=9)
ax.legend(fontsize=7)

# Compute MAE for annotation
gt_pairs = [(r["feats"]["dissolve_frame_free"] * LTX_TEMPORAL_SCALE / VIDEO_FPS,
             gt_dict.get(r["sample_id"]))
            for r in records if gt_dict.get(r["sample_id"]) is not None]
if gt_pairs:
    mae = np.mean([abs(pred - gt) for pred, gt in gt_pairs])
    ax.text(0.05, 0.93, f"MAE = {mae:.2f}s  (n={len(gt_pairs)})",
            transform=ax.transAxes, fontsize=8, color="#333")

plt.tight_layout()
plt.show()


---
## Final View — Video + Velocity Heatmap Per Sample

Each row: the generated video alongside its `pred_mag` heatmap (velocity magnitude: τ × p).

**How to cross-validate:**
1. Play the video. Note the second where the visual cut or dissolve occurs.
2. Convert: `p ≈ floor(time_s × 24 / 8)`.
3. Check if the **yellow dashed line** lands on that frame.
4. Check if the **bright vertical stripe** aligns with it.

**If all three agree** (visual cut ≈ yellow line ≈ bright stripe) → `pred_mag` is a reliable transition predictor.  
**If they disagree** → possible causes: gradual blend (not a hard cut), curvature peak in conditioning region, or the model placed the transition at a frame you can't easily see perceptually.

In [ ]:
# ── Per-sample: video + matching v_pred heatmap side-by-side ─────────────────
import io

all_html = []
all_html.append(
    '<h3 style="font-family:sans-serif">Final View — Video + pred_mag heatmap (per sample)</h3>'
    '<p style="font-family:sans-serif;font-size:12px">'
    'Each row: \U0001F3AC generated video  |  pred_mag (\u03c4 \u00d7 p) heatmap.<br>'
    '\U0001F7E1 dashed=pred p*  \U0001F7E2 solid=GT annotation  '
    'cyan=start-clip  magenta=end-clip</p>'
)

for r in records:
    f    = r["feats"]
    pm   = f["pred_mag"]
    df   = f["dissolve_frame_free"]
    col  = r["color"]
    sid  = r["sample_id"]
    vmax = float(np.percentile(pm, 99))
    gs   = gt_dict.get(sid)
    gp   = gt_latent(sid)

    fig, ax = plt.subplots(figsize=(7, 2.8))
    im = ax.imshow(pm, aspect="auto", cmap="plasma", vmin=0, vmax=vmax,
                   origin="upper", interpolation="nearest",
                   extent=[-0.5, f["F"]-0.5, f["S"]-0.5, -0.5])
    ax.axvline(df, color="#FFC107", lw=2.0, ls="--",
               label=f"pred p*={df} ({df*LTX_TEMPORAL_SCALE/VIDEO_FPS:.2f}s)")
    ax.axvline(K_LAT   - 0.5, color="#00BCD4", lw=1.2, ls="--", alpha=0.8)
    ax.axvline(END_IDX - 0.5, color="#E91E63", lw=1.2, ls="--", alpha=0.8)
    if gp is not None:
        ax.axvline(gp, color="#4CAF50", lw=2.2, ls="-",
                   label=f"GT {gs:.2f}s (p\u2248{gp:.1f})", alpha=0.9)
    ax.set_xlabel("frame p", fontsize=8)
    ax.set_ylabel("step \u03c4", fontsize=8)
    ax.set_xticks(range(f["F"]))
    ax.tick_params(labelsize=6.5)
    ax.legend(fontsize=7, loc="upper right", framealpha=0.8)
    pred_t = df * LTX_TEMPORAL_SCALE / VIDEO_FPS
    err_str = f"  err={abs(pred_t - gs):.2f}s" if gs is not None else ""
    ax.set_title(
        f"pred_mag  [strength={f['dissolve_strength']:.1f}\u00d7  angular_min={f['angular_min']:.3f}]{err_str}",
        fontsize=8.5
    )
    plt.colorbar(im, ax=ax, shrink=0.9, pad=0.02)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode()

    vid_h  = video_html(r["video_path"], width=340, label=r["short_id"], cls=r["cls"])
    plot_h = f'<img src="data:image/png;base64,{img_b64}" style="height:215px;vertical-align:top">'
    all_html.append(
        f'<div style="display:flex;align-items:flex-start;gap:12px;'
        f'margin-bottom:12px;border-bottom:1px solid #ddd;padding-bottom:10px">'
        f'{vid_h}{plot_h}</div>'
    )

display(HTML("\n".join(all_html)))


---
## Next Steps & Research Directions

### Transition time control (highest ROI)
1. **`end_idx` ablation** — Change `end_clip_index` in the config to `{6, 8, 10, 12}`. The `pred_mag` bright stripe should shift with it. Run exp_021 with different values and compare `dissolve_frame_free`.
2. **`z_t` mid-trajectory intervention** — Run Stage-1 to the commit step τ*. Then blend `z_t[:,:,p_target,:,:]` toward a shifted position. Continue denoising. Predict: transition shifts by the same amount.

### Style control (medium ROI)
3. **Guidance scale schedule** — Instead of constant `guidance_scale=4.0`, ramp it up at the (τ, p) region where `pred_mag` peaks. This amplifies text conditioning exactly where the model generates the transition.
4. **Hidden-state probing** — If the Level 3 similarity matrix shows block-diagonal structure for class 6/8 but not class 1/2: train a linear probe to predict "frame is in transition region" from hidden states. This quantifies which block encodes transition structure.

### Immediate validation with this notebook
- Does `dissolve_strength` correlate with semantic class? (summary table + scatter)
- Does `commit_step` differ between easy (class 1) and hard (class 8) transitions?
- In the PCA plots: does frame `p*` appear as an outlier or cluster boundary?
- In the similarity matrix: do class 6/8 show sharper block structure than class 1?
- Cross-validate: play each video and note the perceptual cut time. Does it match `dis_t_s`?

> **Tip:** Add a `gt_dict = {sample_id: float_seconds}` dictionary with your perceptual annotations and plot it alongside `dis_t_s` in the summary scatter for a direct validation.